# VMGP: Variational Auto-Encoder Multi-task Genomic Prediction
## Training & Risultati

Questo notebook usa il modulo `vae/` per training e analisi.

La logica riutilizzabile è stata estratta in moduli Python:

- `vae/experiment.py` — `run_grid`, `run_single` (orchestrazione)
- `vae/plotting.py` — funzioni di visualizzazione
- `vae/attribution.py` — helper SHAP / Random Forest
- `vae/results_io.py` — salvataggio incrementale risultati

Il notebook è ora un livello di orchestrazione/presentazione.

### Struttura del Modulo `vae/`:
- `vae/network.py` - Architettura VMGP
- `vae/lightning_module.py` - Logic training (PyTorch Lightning)
- `vae/data_module.py` - Caricamento e preprocessing dati PLINK
- `vae/utils.py` - Funzioni di utilità (balancing, capping, metriche)
- `vae/experiment.py` - Runner esperimenti con K-Fold CV
- `vae/analysis.py` - Analisi post-training (legacy, non usato qui)

### Esperimento:
Griglia completa di **20 combinazioni**:
- **Min Samples per Razza:** 30, 40, 50, 60, 70 (solo sample reali, NO SMOTE)
- **Dimensione Latent Space:** 32, 64, 96, 128
- **Task:** Solo classificazione breed

---


## 1. Setup e Imports

In [ ]:
import os
import sys
import torch
import pandas as pd
import numpy as np
import pytorch_lightning as pl

# Aggiungi la directory parent al path per importare il modulo vae
sys.path.insert(0, os.path.dirname(os.getcwd()))

# Imposta seed per riproducibilità
pl.seed_everything(42)

from vae import GenomicDataModule, VMGP_LightningSystem
from vae.experiment import run_grid, run_single
from vae.plotting import (
    plot_grid_heatmaps,
    plot_tradeoff,
    plot_trends,
    plot_cm_grid,
    plot_single_cm,
)
from vae.attribution import (
    BreedWrapper,
    aggregate_shap_to_global,
    run_shap_accuracy_curve,
    run_rf_selection,
)

print("✅ Moduli caricati correttamente!")
print(f"🔧 CUDA disponibile: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")


## 2. Configurazione

In [ ]:
# ⚠️ MODIFICA QUESTI PATH CON IL TUO DATASET ⚠️
BED_PATH = "../data/ADAPTmap_genotypeTOP_20160222_full.bed"
METADATA_PATH = "classificazione-capre.csv"

# Hyperparameters
config = {
    # DataModule
    'bed_path': BED_PATH,
    'metadata_path': METADATA_PATH,
    'batch_size': 64,
    'maf_thresh': 0.01,
    'geno_thresh': 0.1,
    'min_samples_per_class': 30,  # Verrà variato negli esperimenti
    'ld_pruning': True,
    'ld_threshold': 0.2,
    'ld_window': 50,
    'val_split': 0.15,
    'test_split': 0.15,

    # Cap samples (solo downsampling, NO dati sintetici)
    'target_samples': 30,  # Verrà variato negli esperimenti

    # Model (VMGP) - Loss weights
    'alpha': 0.5,
    'lr': 0.0001,
    'latent_dim': 32,  # Default, verrà variato negli esperimenti

    # Solo classificatore breed
    'weight_breed': 1.0,
    'weight_continent': 0.0,
    'weight_caseina': 0.0,

    # Training
    'max_epochs': 100,
    'k_folds': 3,  # 3-Fold Cross Validation
    'accelerator': 'gpu' if torch.cuda.is_available() else 'cpu',
    'devices': 1,
}

# Griglia esperimenti
MIN_SAMPLES_RANGE = [30, 40, 50, 60, 70]   # 5 valori
LATENT_DIM_RANGE = [32, 64, 96, 128]        # 4 valori
# Totale: 5 × 4 = 20 combinazioni

print("📋 CONFIGURAZIONE BASE:")
for k, v in config.items():
    print(f"   {k}: {v}")

print(f"\n📊 GRIGLIA ESPERIMENTI:")
print(f"   Min Samples: {MIN_SAMPLES_RANGE}")
print(f"   Latent Dims: {LATENT_DIM_RANGE}")
print(f"   K-Fold CV: {config['k_folds']} fold")
print(f"   Totale combinazioni: {len(MIN_SAMPLES_RANGE) * len(LATENT_DIM_RANGE)}")
print(f"   Task: Solo classificazione breed (NO continente, NO caseina)")
print(f"   Dati: Solo sample reali (NO SMOTE)")

# Conteggio razze disponibili per ogni soglia
print(f"\n📈 RAZZE DISPONIBILI PER SOGLIA (dal dataset ADAPTmap):")
print(f"   ≥30 samples: 34 razze")
print(f"   ≥40 samples: 26 razze")
print(f"   ≥50 samples: 16 razze")
print(f"   ≥60 samples: 12 razze")
print(f"   ≥70 samples: 10 razze")


## 3. Grid Experiment

In [ ]:
# === ESPERIMENTO: GRIGLIA MIN SAMPLES × LATENT DIM ===
# 20 combinazioni totali (5 min_samples × 4 latent_dims)
# Solo breed classification, solo dati reali (cap, NO SMOTE)

grid_results, all_results = run_grid(config, MIN_SAMPLES_RANGE, LATENT_DIM_RANGE)


### 3.1 Grid results / figures

In [ ]:
# === ANALISI GRIGLIA MIN SAMPLES × LATENT DIM ===
df_grid = pd.DataFrame(grid_results)

# Metriche da visualizzare come heatmap (include Kappa per confronto equo)
metrics_to_plot = {
    'Acc_Breed (mean)': 'Breed Accuracy (raw)',
    'Kappa_Breed (mean)': "Cohen's Kappa (chance-adjusted)",
    'Silhouette (mean)': 'Silhouette Score (Clustering)',
    'Generalization (GE) (mean)': 'Generalization (GE)'
}
# Filtra solo le metriche disponibili
metrics_to_plot = {k: v for k, v in metrics_to_plot.items() if k in df_grid.columns}

plot_grid_heatmaps(df_grid, metrics_to_plot)

# Tabella riassuntiva
print("\n📋 TABELLA COMPLETA DEI RISULTATI:")
display_cols = ['Experiment', 'Min_Samples', 'Latent_Dim', 'Num Classes Breed', 'Num SNPs', 'Effective Batch Size']
for m in metrics_to_plot.keys():
    display_cols.append(m)
available_cols = [c for c in display_cols if c in df_grid.columns]
print(df_grid[available_cols].to_string())

# Nota su confronto equo
print("\n⚠️ NOTA SUL CONFRONTO EQUO:")
print("   L'Accuracy raw NON è direttamente comparabile tra soglie diverse")
print("   (classificare 10 razze è più facile che classificarne 34).")
print("   → Usa Cohen's Kappa per un confronto equo (chance-adjusted).")

# Identificazione migliore combinazione
if 'Kappa_Breed (mean)' in df_grid.columns:
    best_idx = df_grid['Kappa_Breed (mean)'].idxmax()
    best = df_grid.loc[best_idx]
    print(f"\n🎯 MIGLIOR COMBINAZIONE (per Cohen's Kappa - confronto equo):")
    print(f"   Min Samples = {best['Min_Samples']}, Latent Dim = {best['Latent_Dim']}")
    print(f"   Kappa: {best['Kappa_Breed (mean)']:.4f}")
    print(f"   Accuracy: {best.get('Acc_Breed (mean)', 'N/A'):.4f}")
    print(f"   Num Razze: {best.get('Num Classes Breed', 'N/A')}")

if 'Acc_Breed (mean)' in df_grid.columns:
    best_acc_idx = df_grid['Acc_Breed (mean)'].idxmax()
    best_acc = df_grid.loc[best_acc_idx]
    print(f"\n🎯 MIGLIOR COMBINAZIONE (per Accuracy raw):")
    print(f"   Min Samples = {best_acc['Min_Samples']}, Latent Dim = {best_acc['Latent_Dim']}")
    print(f"   Accuracy: {best_acc['Acc_Breed (mean)']:.4f}")
    print(f"   Num Razze: {best_acc.get('Num Classes Breed', 'N/A')}")


### 3.2 Trade-off analysis

In [ ]:
# === ANALISI TRADE-OFF: NUMERO RAZZE vs PERFORMANCE ===
plot_tradeoff(df_grid, MIN_SAMPLES_RANGE, LATENT_DIM_RANGE)

# Statistiche riassuntive
print("\n📊 STATISTICHE PER MIN_SAMPLES (media su tutte le latent dim):")
summary_cols = {'Num Classes Breed': 'first'}
if 'Acc_Breed (mean)' in df_grid.columns:
    summary_cols['Acc_Breed (mean)'] = ['mean', 'std', 'max']
if 'Kappa_Breed (mean)' in df_grid.columns:
    summary_cols['Kappa_Breed (mean)'] = ['mean', 'std', 'max']
summary = df_grid.groupby('Min_Samples').agg(summary_cols).round(4)
print(summary.to_string())

print("\n📊 STATISTICHE PER LATENT DIM (media su tutti i min_samples):")
dim_cols = {}
if 'Acc_Breed (mean)' in df_grid.columns:
    dim_cols['Acc_Breed (mean)'] = ['mean', 'std', 'max']
if 'Kappa_Breed (mean)' in df_grid.columns:
    dim_cols['Kappa_Breed (mean)'] = ['mean', 'std', 'max']
summary_dim = df_grid.groupby('Latent_Dim').agg(dim_cols).round(4)
print(summary_dim.to_string())


### 3.3 Trend per latent dim

In [ ]:
# === LINEE DI TENDENZA PER LATENT DIM ===
plot_trends(df_grid, MIN_SAMPLES_RANGE, LATENT_DIM_RANGE)


### 3.4 Conclusioni

In [ ]:
# === CONCLUSIONI GRIGLIA ESPERIMENTI ===
print("🎯 ANALISI DEI RISULTATI (20 combinazioni con 3-Fold CV)")
print("="*60)

if 'Kappa_Breed (mean)' in df_grid.columns:
    # Top 5 per Kappa (confronto equo)
    top5_kappa = df_grid.nlargest(5, 'Kappa_Breed (mean)')
    print("\n🏆 TOP 5 CONFIGURAZIONI (per Cohen's Kappa - confronto equo):")
    for i, (_, row) in enumerate(top5_kappa.iterrows(), 1):
        medal = ["🥇", "🥈", "🥉", "4️⃣", "5️⃣"][i-1]
        print(f"   {medal} Samples={int(row['Min_Samples'])}, LatDim={int(row['Latent_Dim'])} "
              f"→ Kappa={row['Kappa_Breed (mean)']:.4f}, Acc={row.get('Acc_Breed (mean)', 0):.4f} "
              f"({int(row.get('Num Classes Breed', 0))} razze)")

if 'Acc_Breed (mean)' in df_grid.columns:
    top5 = df_grid.nlargest(5, 'Acc_Breed (mean)')
    print("\n🏆 TOP 5 CONFIGURAZIONI (per Accuracy raw):")
    for i, (_, row) in enumerate(top5.iterrows(), 1):
        medal = ["🥇", "🥈", "🥉", "4️⃣", "5️⃣"][i-1]
        print(f"   {medal} Samples={int(row['Min_Samples'])}, LatDim={int(row['Latent_Dim'])} "
              f"→ Acc={row['Acc_Breed (mean)']:.4f} ({int(row.get('Num Classes Breed', 0))} razze)")

print("\n" + "="*60)
print("📝 NOTE:")
print("   • Tutti gli esperimenti usano SOLO sample reali (NO dati sintetici)")
print("   • Task: solo classificazione breed")
print("   • Le razze con più sample del minimo vengono cappate (downsampled)")
print("   • Cohen's Kappa è la metrica preferita per confronto equo tra soglie diverse")
print("   • 3-Fold Cross Validation per ogni esperimento (media ± std su 3 fold)")
print("   • Batch size fisso per tutti gli esperimenti")
print("   • Le confusion matrix sono in results/confusion_matrices/")

# Salva CSV con tutti i risultati (escludi colonne non serializzabili)
os.makedirs('results', exist_ok=True)
export_cols = [c for c in df_grid.columns if c not in ['Confusion Matrix Breed', 'Class Names Breed']]
df_grid[export_cols].to_csv('results/grid_experiment_results.csv', index=False)
print("\n💾 Risultati salvati in results/grid_experiment_results.csv")


### 3.5 Confusion matrices

In [ ]:
# === CONFUSION MATRIX PER OGNI ESPERIMENTO ===
plot_cm_grid(grid_results, MIN_SAMPLES_RANGE, LATENT_DIM_RANGE)

print("✅ Confusion matrix visualizzate per tutte le 20 combinazioni")


## 4. Single Experiment

In [ ]:
# ═══════════════════════════════════════════════════
# CONFIGURA QUI
# ═══════════════════════════════════════════════════
SINGLE_MIN_SAMPLES = 30    # Numero minimo di campioni per razza
SINGLE_LATENT_DIM  = 96    # Dimensione del latent space
# ═══════════════════════════════════════════════════

print(f"Configurazione singolo esperimento:")
print(f"   Min Samples: {SINGLE_MIN_SAMPLES}")
print(f"   Latent Dim:  {SINGLE_LATENT_DIM}")


In [ ]:
# === TRAINING SINGOLO ESPERIMENTO ===
single_result, exp_name = run_single(config, SINGLE_MIN_SAMPLES, SINGLE_LATENT_DIM)


### 4.1 Single-experiment results

In [ ]:
# === RISULTATI SINGOLO ESPERIMENTO ===
print(f"📊 RISULTATI: {exp_name}")
print(f"{'='*60}")
print(f"   Razze:       {single_result.get('Num Classes Breed', '?')}")
print(f"   SNPs:        {single_result.get('Num SNPs', '?')}")
print(f"   Best Fold:   {single_result.get('Best Fold', '?')}")

# Stampa tutte le metriche (mean ± std)
print(f"\n   Metriche (mean ± std su {config['k_folds']} fold):")
for key in sorted(single_result.keys()):
    if key.endswith('(mean)'):
        std_key = key.replace('(mean)', '(std)')
        mean_val = single_result[key]
        std_val = single_result.get(std_key, 0)
        metric_name = key.replace(' (mean)', '')
        print(f"   {metric_name:30s}: {mean_val:.4f} ± {std_val:.4f}")

# Confusion Matrix
plot_single_cm(single_result, exp_name)


## 5. SHAP analysis

> ⚠️ **Nota metodologica (intenzionalmente NON corretta qui).**
> L'attribuzione SHAP/RF è calcolata sull'**intero dataset** (non solo
> train): questo è un noto problema di leakage che verrà corretto in una
> fase successiva.

Studio analogo al modello Contrastivo: verificare se i Top-% SNP identificati
dagli SHAP del classificatore breed del VMGP sono sufficienti per classificare
la razza meglio di una selezione casuale.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEZIONE 5 — SHAP SNP Pre-selection (protocollo paper VMGP)
# ═══════════════════════════════════════════════════════════════
import shap
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from pytorch_lightning.callbacks import EarlyStopping

# ── CONFIG (riusa il DataModule già caricato per min_samples=30) ──
SHAP_MIN_SAMPLES = 30
SHAP_LATENT_DIM  = 96       # usa la tua configurazione migliore
SHAP_EPOCHS      = 100
SEED             = 42
os.makedirs('results/shap_vmgp', exist_ok=True)

# ── 1. Carica DataModule dedicato per SHAP ────────────────────────
dm = GenomicDataModule(
    bed_path=BED_PATH,
    metadata_path=METADATA_PATH,
    batch_size=64,
    maf_thresh=0.01,
    geno_thresh=0.1,
    min_samples_per_class=SHAP_MIN_SAMPLES,
    ld_pruning=True,
    ld_threshold=0.2,
    ld_window=50,
    val_split=0.15,
    test_split=0.15,
    use_breed=True,
    use_continent=False,
    use_caseina=False,
    use_attitudine=False
)
dm.setup()

X_all = dm.X_processed
y_all = dm.y_breed_processed
print(f"Dataset completo: {X_all.shape[0]} campioni | {X_all.shape[1]} SNP | {dm.num_classes_breed} razze")

# ── 2. Split 10% ps / 90% validation (stratificato per razza) ────
ps_idx, val90_idx = train_test_split(
    np.arange(len(X_all)),
    test_size=0.90, stratify=y_all, random_state=SEED
)
X_ps,  y_ps  = X_all[ps_idx],     y_all[ps_idx]
X_90,  y_90  = X_all[val90_idx],  y_all[val90_idx]
print(f"ps (10%): {len(X_ps)} campioni | val90: {len(X_90)} campioni")

# ── 4. Training modello ps ────────────────────────────────────────
print("\n🔧 Training modello ps (10%)...")

# Dummy y_cont/cas/att (non usati — breed_only)
y_dummy = np.zeros(len(y_ps), dtype=np.int64)

ps_ds = TensorDataset(
    torch.FloatTensor(X_ps),
    torch.LongTensor(y_ps),
    torch.LongTensor(y_dummy),  # continent
    torch.LongTensor(y_dummy),  # caseina
    torch.LongTensor(y_dummy),  # attitudine
)
# Split interno 85/15 per l'early stopping
n_val_ps = max(int(0.15 * len(ps_ds)), 1)
n_train_ps = len(ps_ds) - n_val_ps
ps_train_ds, ps_val_ds = torch.utils.data.random_split(
    ps_ds, [n_train_ps, n_val_ps],
    generator=torch.Generator().manual_seed(SEED)
)

ps_train_loader = DataLoader(ps_train_ds, batch_size=32, shuffle=True,
                              num_workers=4, pin_memory=True, drop_last=True)
ps_val_loader   = DataLoader(ps_val_ds,   batch_size=32,
                              num_workers=4, pin_memory=True)

model_ps = VMGP_LightningSystem(
    num_snps=dm.num_snps,
    num_classes_breed=dm.num_classes_breed,
    num_classes_continent=0,
    num_classes_caseina=0,
    num_classes_attitudine=0,
    alpha=0.5, lr=0.0001,
    use_breed=True, use_continent=False, use_caseina=False, use_attitudine=False,
    weight_breed=1.0, weight_continent=0.0, weight_caseina=0.0, weight_attitudine=0.0,
    latent_dim=SHAP_LATENT_DIM
)

early_stop = EarlyStopping(monitor='val_loss', patience=15, mode='min', verbose=False)
trainer_ps = pl.Trainer(
    precision="16-mixed", max_epochs=SHAP_EPOCHS,
    accelerator='gpu', devices=1,
    callbacks=[early_stop],
    enable_progress_bar=True, enable_model_summary=False,
    enable_checkpointing=False, logger=False
)
trainer_ps.fit(model_ps, ps_train_loader, ps_val_loader)
print("✅ Training ps completato.")

# ── 5. Calcolo SHAP values ─────────────────────────────────────────
wrapper = BreedWrapper(model_ps).eval().to('cuda')


X_ps_t = torch.FloatTensor(X_ps).to('cuda')
# Background: 100 campioni random dal ps (come da paper)
np.random.seed(SEED)
bg_idx  = np.random.choice(len(X_ps), min(100, len(X_ps)), replace=False)
X_bg_t  = X_ps_t[bg_idx]

print(f"\n🔍 Calcolo SHAP su {len(X_ps)} campioni ps...")
explainer   = shap.GradientExplainer(wrapper, X_bg_t)
shap_values = explainer.shap_values(X_ps_t)   # list o ndarray, dipende dalla versione SHAP

# ── 6. Aggregazione robusta (helper estratto) ─────────────────────
N_SNPS = X_all.shape[1]
shap_global = aggregate_shap_to_global(shap_values, N_SNPS)

np.save('results/shap_vmgp/shap_global.npy', shap_global)
print(f"✅ SHAP global salvato. Shape: {shap_global.shape}")

# Top 10 SNP
top10 = np.argsort(shap_global)[::-1][:10]
print("\nTop 10 SNP per SHAP aggregato:")
for r, idx in enumerate(top10, 1):
    print(f"  {r:2d}. SNP[{int(idx)}] = {float(shap_global[idx]):.6f}")

# ── 7. Curva accuracy per soglia di SNP (helper estratto) ─────────
df_shap = run_shap_accuracy_curve(shap_global, X_90, y_90, seed=SEED)


## 6. RF analysis

In [ ]:
# Random forest approach (helper estratto)
df_rf = run_rf_selection(X_all, y_all, X_ps, y_ps, X_90, y_90, seed=SEED)


## 7. Interpretation

Analisi e conclusioni qualitative sui risultati delle sezioni precedenti
(vanno inserite qui in base agli output di training).
